In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/src/small_bench/checkpoints/pre_cell_2.pickle

In [ ]:
%%cudf.pandas.profile
### cell 2 ###

# Optimized for cudf: batch concat and GPU operations
def col_name_corrections(df, correction_pair):
    # rename only if needed
    if set(df.columns).intersection(correction_pair):
        return df.rename(columns=correction_pair)
    return df

import os, re

input_dir = '/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input'
# gather all file paths
file_paths = [os.path.join(d, f)
              for d, _, files in os.walk(input_dir)
              for f in files]

mobile_chunks = []
broadband_chunks = []
for path in file_paths:
    fname = os.path.basename(path)
    # read CSV on GPU via cudf-pandas extension
    df = pd.read_csv(path, thousands=',').convert_dtypes()
    df = col_name_corrections(df, {'Number of Record': 'Number of Records'})
    # extract year & quarter
    year = int(re.search(r'year_(.*)_quarter', fname).group(1))
    quarter = int(re.search(r'quarter_(.*)\.csv', fname).group(1))
    df['Year'] = year
    df['Quarter'] = quarter
    # partition into lists
    if 'mobile' in fname:
        mobile_chunks.append(df)
    else:
        broadband_chunks.append(df)
# single concat per category (GPU)
Mobile_df = pd.concat(mobile_chunks, ignore_index=True)
Broadband_df = pd.concat(broadband_chunks, ignore_index=True)

print(Broadband_df.shape)
print(Mobile_df.shape)
# cast & sort on GPU, return new DF (avoids CPU inplace)
Mobile_df = Mobile_df.astype({'Year':'int64','Quarter':'int64'})\
                   .sort_values(by=['Year','Quarter'], ascending=[True,True], ignore_index=True)
Broadband_df = Broadband_df.astype({'Year':'int64','Quarter':'int64'})\
                       .sort_values(by=['Year','Quarter'], ascending=[True,True], ignore_index=True)